# MCP Layer

In [1]:
import logging
import os
import base64
import json

from pathlib import Path
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional, Dict, Any, Literal

from pinecone import Pinecone
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

# Load API key from .env file
load_dotenv()
client = OpenAI()
openai_api_key = os.getenv("OPENAI_API_KEY")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("orlando-zoning-index")

### Pydantic Schemas for Input/Output

In [2]:
class ZoningQueryInput(BaseModel):
    """Input schema for zoning law queries"""
    query: str = Field(
        ..., 
        description="Natural language question about Orlando zoning laws",
        examples=["What are the setback requirements for R-1A zones?"]
    )
    top_k: int = Field(
        default=5, 
        description="Number of relevant document chunks to retrieve",
        ge=1,
        le=20
    )
    similarity_threshold: float = Field(
        default=0.7,
        description="Minimum similarity score for retrieved chunks (0-1)",
        ge=0.0,
        le=1.0
    )

class SourceCitation(BaseModel):
    """Individual source citation with metadata"""
    source_document: str = Field(..., description="Name of the source document")
    page_number: Optional[int] = Field(None, description="Page number in source document")
    chunk_id: str = Field(..., description="Unique identifier for this chunk")
    chunk_text: str = Field(..., description="Relevant text excerpt from the document")
    similarity_score: float = Field(..., description="Cosine similarity score (0-1)")

class ZoningQueryOutput(BaseModel):
    """Output schema for zoning law queries"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "query": "What are setback requirements for R-1A?",
                "answer": "For R-1A residential zones, the minimum setback requirements are...",
                "sources": [
                    {
                        "source_document": "orlando_zoning_code.docx",
                        "page_number": 42,
                        "chunk_id": "chunk_0123",
                        "chunk_text": "Setback requirements for R-1A zones...",
                        "similarity_score": 0.89
                    }
                ],
                "chunks_retrieved": 5,
                "total_tokens_used": 450
            }
        }
    )
    
    query: str = Field(..., description="Original query for reference")
    answer: str = Field(..., description="Generated answer based on retrieved zoning law context")
    sources: List[SourceCitation] = Field(..., description="Source citations supporting the answer")
    chunks_retrieved: int = Field(..., description="Number of chunks retrieved from vector store")
    total_tokens_used: Optional[int] = Field(None, description="Total tokens used in generation")

### Tool Class

In [3]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ZoningLawTool:
    """
    MCP-compatible tool for querying Orlando zoning law knowledge base.
    
    This tool uses RAG (Retrieval-Augmented Generation) to answer questions
    about Orlando municipal zoning codes by:
    1. Retrieving relevant document chunks from Pinecone vector store
    2. Generating accurate answers using OpenAI based on retrieved context
    """
    
    # MCP metadata for LLM discovery
    name: str = "zoning_law_query"
    description: str = """
    Query the Orlando municipal zoning code knowledge base using natural language.
    
    Use this tool when you need information about:
    - Zoning classifications (R-1A, R-2, C-1, etc.)
    - Setback requirements and lot dimensions
    - Permitted and conditional uses for zones
    - Density and height restrictions
    - Parking requirements
    - Variance and special exception rules
    - Any other Orlando zoning regulations
    
    The tool retrieves relevant sections from official zoning documents and 
    provides accurate, citation-backed answers.
    """
    
    def __init__(
        self,
        pinecone_api_key: Optional[str] = None,
        openai_api_key: Optional[str] = None,
        index_name: str = "orlando-zoning-laws",
        embedding_model: str = "text-embedding-3-small",
        generation_model: str = "gpt-4o-mini",
        namespace: str = "__default__"
    ):
        """
        Initialize the Zoning Law Tool.
        
        Args:
            pinecone_api_key: Pinecone API key (reads from env if not provided)
            openai_api_key: OpenAI API key (reads from env if not provided)
            index_name: Name of the Pinecone index
            embedding_model: OpenAI embedding model to use
            generation_model: OpenAI chat model for answer generation
            namespace: Pinecone namespace for zoning documents
        """
        # Get API keys from environment if not provided
        self.pinecone_api_key = pinecone_api_key or os.getenv("PINECONE_API_KEY")
        self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        
        if not self.pinecone_api_key:
            raise ValueError("Pinecone API key not found. Set PINECONE_API_KEY environment variable.")
        if not self.openai_api_key:
            raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable.")
        
        # Initialize clients
        logger.info("Initializing Pinecone client...")
        self.pc = Pinecone(api_key=self.pinecone_api_key)
        self.index = self.pc.Index(index_name)
        self.namespace = namespace
        
        logger.info("Initializing OpenAI client...")
        self.openai_client = OpenAI(api_key=self.openai_api_key)
        
        # Store model names
        self.embedding_model = embedding_model
        self.generation_model = generation_model
        
        logger.info(f"ZoningLawTool initialized successfully with index '{index_name}'")
    
    def _create_embedding(self, text: str) -> List[float]:
        """Create embedding vector for query text."""
        try:
            response = self.openai_client.embeddings.create(
                model=self.embedding_model,
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            logger.error(f"Error creating embedding: {e}")
            raise
    
    def _retrieve_chunks(
        self, 
            query: str, 
        top_k: int = 5,
        similarity_threshold: float = 0.7) -> List[Dict[str, Any]]:
        """
        Retrieve relevant chunks from Pinecone with adaptive threshold.
    
        Args:
            query: Search query
            top_k: Number of chunks to retrieve
            similarity_threshold: Minimum similarity score
            
        Returns:
            List of matching chunks with metadata
        """
        try:
            # Create query embedding
            query_embedding = self._create_embedding(query)
            
            # Search Pinecone
            results = self.index.query(
                vector=query_embedding,
                top_k=top_k,
                namespace=self.namespace,
                include_metadata=True
            )
            
            # Filter by similarity threshold and format results
            chunks = []
            for match in results.matches:
                if match.score >= similarity_threshold:
                    chunks.append({
                        "id": match.id,
                        "score": match.score,
                        "metadata": match.metadata,
                        "text": match.metadata.get("text", "")
                    })
            
            # ADAPTIVE FALLBACK: If no chunks found with high threshold, try lower
            if not chunks and similarity_threshold > 0.5:
                logger.warning(
                    f"No chunks found with threshold {similarity_threshold}. "
                    f"Retrying with threshold 0.5..."
                )
                fallback_threshold = 0.5
                for match in results.matches:
                    if match.score >= fallback_threshold:
                        chunks.append({
                            "id": match.id,
                            "score": match.score,
                            "metadata": match.metadata,
                            "text": match.metadata.get("text", "")
                        })
                
                if chunks:
                    logger.info(
                        f"Retrieved {len(chunks)} chunks with fallback threshold {fallback_threshold}"
                    )
            
            # Final logging
            if chunks:
                logger.info(f"Retrieved {len(chunks)} chunks (threshold: {similarity_threshold})")
            else:
                logger.warning(f"No relevant chunks found for query: '{query}'")
            
            return chunks
            
        except Exception as e:
            logger.error(f"Error retrieving chunks: {e}")
            raise
    
    def _generate_answer(
        self, 
        query: str, 
        chunks: List[Dict[str, Any]]
    ) -> tuple[str, int]:
        """
        Generate answer using OpenAI based on retrieved chunks.
        
        Args:
            query: Original user query
            chunks: Retrieved document chunks
            
        Returns:
            Tuple of (generated_answer, tokens_used)
        """
        if not chunks:
            return "No relevant information found in the Orlando zoning code database for this query.", 0
        
        # Build context from chunks
        context_parts = []
        for i, chunk in enumerate(chunks, 1):
            source = chunk["metadata"].get("source", "Unknown")
            page = chunk["metadata"].get("page", "N/A")
            text = chunk["text"]
            context_parts.append(
                f"[Source {i}: {source}, Page {page}]\n{text}"
            )
        
        context = "\n\n".join(context_parts)
        
        # Create prompt
        system_prompt = """You are an expert on Orlando, Florida zoning laws and regulations. 
        
        Your role is to provide accurate, detailed answers about zoning codes based ONLY on the 
        provided context from official Orlando municipal code documents.

        Guidelines:
        - Answer directly and clearly
        - Cite specific sections or requirements when mentioned in the context
        - If the context doesn't fully answer the question, acknowledge what information is available
        - Use professional but accessible language
        - Include relevant details like setback distances, lot sizes, permitted uses, etc.
        - Do NOT make up information not present in the context"""

        user_prompt = f"""Based on the following excerpts from Orlando zoning code documents, 
        please answer this question:

        QUESTION: {query}

        CONTEXT FROM ZONING DOCUMENTS:
        {context}

        ANSWER:"""

        try:
            response = self.openai_client.chat.completions.create(
                model=self.generation_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,  # Low temperature for factual responses
                max_tokens=800
            )
            
            answer = response.choices[0].message.content
            tokens_used = response.usage.total_tokens
            
            return answer, tokens_used
            
        except Exception as e:
            logger.error(f"Error generating answer: {e}")
            raise
    
    def __call__(
        self, 
        query: str, 
        top_k: int = 5,
        similarity_threshold: float = 0.7
    ) -> ZoningQueryOutput:
        """
        Main entry point for the tool. Execute a zoning law query.
        
        Args:
            query: Natural language question about zoning laws
            top_k: Number of document chunks to retrieve (1-20)
            similarity_threshold: Minimum similarity score (0-1)
            
        Returns:
            ZoningQueryOutput with answer and source citations
        """
        logger.info(f"Executing zoning query: '{query}'")
        
        try:
            # Retrieve relevant chunks
            chunks = self._retrieve_chunks(query, top_k, similarity_threshold)
            
            # Generate answer
            answer, tokens_used = self._generate_answer(query, chunks)
            
            # Format source citations
            sources = []
            for chunk in chunks:
                citation = SourceCitation(
                    source_document=chunk["metadata"].get("source", "Unknown"),
                    page_number=chunk["metadata"].get("page"),
                    chunk_id=chunk["id"],
                    chunk_text=chunk["text"][:300] + "..." if len(chunk["text"]) > 300 else chunk["text"],
                    similarity_score=round(chunk["score"], 3)
                )
                sources.append(citation)
            
            # Build output
            output = ZoningQueryOutput(
                query=query,
                answer=answer,
                sources=sources,
                chunks_retrieved=len(chunks),
                total_tokens_used=tokens_used
            )
            
            logger.info(f"Query completed successfully. Retrieved {len(chunks)} chunks, used {tokens_used} tokens")
            return output
            
        except Exception as e:
            logger.error(f"Error executing zoning query: {e}")
            raise
    
    def get_tool_schema(self) -> Dict[str, Any]:
        """
        Return JSON schema for MCP protocol compatibility.
        This allows LLMs to understand how to call this tool.
        """
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": ZoningQueryInput.model_json_schema(),
            "output_schema": ZoningQueryOutput.model_json_schema()
        }

In [4]:
# Initialize the tool
zoning_tool = ZoningLawTool(
    index_name="orlando-zoning-index",
    namespace="__default__"
)

INFO:__main__:Initializing Pinecone client...
INFO:__main__:Initializing OpenAI client...
INFO:__main__:ZoningLawTool initialized successfully with index 'orlando-zoning-index'


In [5]:
# Simple query
result = zoning_tool(
    query="What size tree requires a permit?"
)

print(f"Answer: {result.answer}\n")
print(f"Sources used: {result.chunks_retrieved}")
for i, source in enumerate(result.sources, 1):
    print(f"\n[{i}] {source.source_document} (Page {source.page_number})")
    print(f"    Similarity: {source.similarity_score}")
    print(f"    Text: {source.chunk_text[:150]}...")

INFO:__main__:Executing zoning query: 'What size tree requires a permit?'
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Retrieved 5 chunks with fallback threshold 0.5
INFO:__main__:Retrieved 5 chunks (threshold: 0.7)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Query completed successfully. Retrieved 5 chunks, used 1097 tokens


Answer: A tree requires a permit for removal if it has a diameter at breast height (dbh) of 10 inches or larger, measured at a height of 4.5 feet above the ground. This requirement is outlined in multiple sections of the Orlando zoning code, specifically in Sec. 60.209, Sec. 65.640, and Sec. 60.207. It is prohibited to remove such trees without first obtaining a Tree Removal Permit, unless exempted by Florida Statutes.

Sources used: 5

[1] Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx (Page None)
    Similarity: 0.671
    Text: Sec. 60.209. General Requirements.
(a)	Tree Removal Permit Required. Removal of non-exempt existing 10" dbh or larger trees shall be prohibited withou...

[2] Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (Page None)
    Similarity: 0.669
    Text: 6F. TREE PERMITS...

[3] Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (Page None)
    Similarity: 0.643
    Text: Sec. 65.640. Tree Removal Permit.
When a Tree Removal Permit is Required. Unless exempt by se

In [6]:
# Example 2: Query with custom parameters
result = zoning_tool(
    query="What are permitted uses in commercial zones?",
    top_k=10,
    similarity_threshold=0.6
)

INFO:__main__:Executing zoning query: 'What are permitted uses in commercial zones?'
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Retrieved 5 chunks (threshold: 0.6)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Query completed successfully. Retrieved 5 chunks, used 1455 tokens


In [7]:
# Access the tool schema (for MCP registration)
schema = zoning_tool.get_tool_schema()
print(schema)

{'name': 'zoning_law_query', 'description': '\n    Query the Orlando municipal zoning code knowledge base using natural language.\n\n    Use this tool when you need information about:\n    - Zoning classifications (R-1A, R-2, C-1, etc.)\n    - Setback requirements and lot dimensions\n    - Permitted and conditional uses for zones\n    - Density and height restrictions\n    - Parking requirements\n    - Variance and special exception rules\n    - Any other Orlando zoning regulations\n\n    The tool retrieves relevant sections from official zoning documents and \n    provides accurate, citation-backed answers.\n    ', 'input_schema': {'description': 'Input schema for zoning law queries', 'properties': {'query': {'description': 'Natural language question about Orlando zoning laws', 'examples': ['What are the setback requirements for R-1A zones?'], 'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'description': 'Number of relevant document chunks to retrieve', 'maximum': 20, 'm

In [8]:
# Convert output to dict for logging or API responses
output_dict = result.model_dump()
print(output_dict)

{'query': 'What are permitted uses in commercial zones?', 'answer': 'In commercial zones, the permitted uses include a variety of non-residential activities. Based on the provided excerpts from the Orlando zoning code documents, here are some key points regarding permitted uses in commercial zones:\n\n1. **Hotel/Motel/Timeshare Uses**: These are permitted in Zones A and B with appropriate controls.\n\n2. **Retail Uses**: All retail uses must front on a major thoroughfare, ensuring visibility and accessibility.\n\n3. **Indoor Recreation**: Light indoor recreation uses over 5,000 square feet must be located in stand-alone buildings to minimize impacts on neighboring uses.\n\n4. **Conditional Uses**: Certain uses may require a Conditional Use Permit, especially if they are located within 300 feet of a residential zoning district.\n\n5. **Vocational Schools**: These are classified as permitted, conditional, or prohibited based on the most intense trade taught, as determined by the Zoning O

# Vision Discovery

### Pydantic Schemas for Input and Output AND Tool Class

In [9]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class PropertyDamageInput(BaseModel):
    """Input schema for property damage assessment"""
    image_path: Optional[str] = Field(
        None,
        description="Path to a new property damage image file for analysis"
    )
    search_query: Optional[str] = Field(
        None,
        description="Text query to search existing damage assessments (e.g., 'water damage in ceiling')"
    )
    mode: Literal["analyze", "search"] = Field(
        default="analyze",
        description="Mode: 'analyze' for new image analysis, 'search' for finding similar existing damage"
    )
    top_k: int = Field(
        default=3,
        description="Number of similar damage cases to retrieve (search mode only)",
        ge=1,
        le=10
    )

class PropertyDamageOutput(BaseModel):
    """Output schema for property damage assessment"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "file_name": "damage_04.jpg",
                "damage_type": "Ceiling Water Stain",
                "affected_system": "plumbing",
                "location": "Master Bedroom",
                "severity": 7,
                "urgency": "immediate",
                "visible_area_affected": "medium",
                "estimated_repair_cost": "$800-$2000",
                "secondary_damage_risk": "mold growth, ceiling collapse",
                "maintenance_summary": "Active roof leak or plumbing failure likely."
            }
        }
    )
    
    file_name: str = Field(..., description="Name of the image file")
    damage_type: str = Field(..., description="Type of damage observed")
    affected_system: Literal["roofing", "plumbing", "hvac", "electrical", "structural", "cosmetic"] = Field(
        ..., 
        description="Building system affected by the damage"
    )
    location: str = Field(..., description="Likely location in property")
    severity: int = Field(
        ..., 
        description="Severity score from 1-10",
        ge=1,
        le=10
    )
    urgency: Literal["immediate", "short-term", "monitor"] = Field(
        ...,
        description="How urgently repairs are needed"
    )
    visible_area_affected: Literal["small", "medium", "large"] = Field(
        ...,
        description="Size of affected area"
    )
    estimated_repair_cost: str = Field(
        ...,
        description="Rough cost range (e.g., $500-$1500)"
    )
    secondary_damage_risk: str = Field(
        ...,
        description="Potential consequences if untreated"
    )
    maintenance_summary: str = Field(
        ...,
        description="Brief expert assessment and recommended action"
    )
    similarity_score: Optional[float] = Field(
        None,
        description="Similarity score if retrieved from search (0-1)"
    )


class VisionSearchResult(BaseModel):
    """Result for search mode containing multiple similar damage cases"""
    query: str = Field(..., description="Original search query")
    results: List[PropertyDamageOutput] = Field(..., description="List of similar damage cases")
    results_count: int = Field(..., description="Number of results returned")


class VisionTool:
    """
    MCP-compatible tool for property damage assessment using GPT-4o-mini vision
    and semantic search over existing damage cases.
    
    Two modes:
    1. Analyze Mode: Assess a new property image for damage
    2. Search Mode: Find similar damage cases from existing assessments
    """
    
    # MCP metadata for LLM discovery
    name: str = "property_damage_assessment"
    description: str = """
    Analyze property damage images or search existing damage assessments.
    
    Two modes available:
    
    ANALYZE MODE (default):
    - Provide an image_path to analyze a new property damage image
    - Returns detailed damage assessment including severity, cost, urgency
    - Identifies affected systems (roofing, plumbing, HVAC, etc.)
    
    SEARCH MODE:
    - Provide a search_query to find similar damage cases from database
    - Returns multiple matching cases with similarity scores
    - Useful for comparing damage patterns or finding precedents
    
    Examples:
    - Analyze: image_path="damage_photo.jpg", mode="analyze"
    - Search: search_query="water damage ceiling", mode="search", top_k=5
    """
    
    # Damage assessment schema for structured output
    _damage_schema = {
        "name": "property_damage_assessment",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "file_name": {"type": "string"},
                "damage_type": {"type": "string"},
                "affected_system": {
                    "type": "string",
                    "enum": ["roofing", "plumbing", "hvac", "electrical", "structural", "cosmetic"]
                },
                "location": {"type": "string"},
                "severity": {"type": "integer"},
                "urgency": {
                    "type": "string",
                    "enum": ["immediate", "short-term", "monitor"]
                },
                "visible_area_affected": {
                    "type": "string",
                    "enum": ["small", "medium", "large"]
                },
                "estimated_repair_cost": {"type": "string"},
                "secondary_damage_risk": {"type": "string"},
                "maintenance_summary": {"type": "string"}
            },
            "required": [
                "file_name", "damage_type", "affected_system", "location",
                "severity", "urgency", "visible_area_affected",
                "estimated_repair_cost", "secondary_damage_risk", "maintenance_summary"
            ],
            "additionalProperties": False
        }
    }
    
    def __init__(
        self,
        openai_api_key: Optional[str] = None,
        pinecone_api_key: Optional[str] = None,
        index_name: str = "orlando-zoning-index",
        namespace: str = "property-damage",
        embedding_model: str = "text-embedding-3-small",
        vision_model: str = "gpt-4o-mini",
        max_tokens: int = 500
    ):
        """
        Initialize the Vision Tool.
        
        Args:
            openai_api_key: OpenAI API key (reads from env if not provided)
            pinecone_api_key: Pinecone API key (reads from env if not provided)
            index_name: Name of the Pinecone index
            namespace: Pinecone namespace for property damage embeddings
            embedding_model: OpenAI embedding model
            vision_model: Vision model to use (default: gpt-4o-mini)
            max_tokens: Maximum tokens for vision response
        """
        # Get API keys from environment if not provided
        self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        self.pinecone_api_key = pinecone_api_key or os.getenv("PINECONE_API_KEY")
        
        if not self.openai_api_key:
            raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable.")
        if not self.pinecone_api_key:
            raise ValueError("Pinecone API key not found. Set PINECONE_API_KEY environment variable.")
        
        # Initialize clients
        logger.info("Initializing OpenAI client for vision analysis...")
        self.openai_client = OpenAI(api_key=self.openai_api_key)
        
        logger.info("Initializing Pinecone client...")
        self.pc = Pinecone(api_key=self.pinecone_api_key)
        self.index = self.pc.Index(index_name)
        self.namespace = namespace
        
        # Store model settings
        self.embedding_model = embedding_model
        self.vision_model = vision_model
        self.max_tokens = max_tokens
        
        logger.info(f"VisionTool initialized successfully")
    
    def _encode_image(self, image_path: Path) -> str:
        """Read image file and return base64 encoded string."""
        try:
            with open(image_path, "rb") as f:
                return base64.b64encode(f.read()).decode("utf-8")
        except Exception as e:
            logger.error(f"Error encoding image: {e}")
            raise
    
    def _create_embedding(self, text: str) -> List[float]:
        """Create embedding vector for text query."""
        try:
            response = self.openai_client.embeddings.create(
                model=self.embedding_model,
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            logger.error(f"Error creating embedding: {e}")
            raise
    
    def _analyze_new_image(self, image_path: Path, file_name: Optional[str] = None) -> PropertyDamageOutput:
        """
        Analyze a new image using GPT-4o-mini vision.
        
        Args:
            image_path: Path to the image file
            file_name: Optional override for file name
            
        Returns:
            PropertyDamageOutput with damage assessment
        """
        try:
            # Encode image to base64
            base64_image = self._encode_image(image_path)
            
            # Use provided file name or extract from path
            img_file_name = file_name or image_path.name
            
            # Create vision prompt
            response = self.openai_client.chat.completions.create(
                model=self.vision_model,
                messages=[
                    {
                        "role": "system",
                        "content": """You are an expert property inspector assessing damage for real estate valuation.
                        
                        Analyze the image and provide a detailed damage assessment.
                        
                        Be specific about damage type, severity, and repair recommendations.
                        
                        Base severity scores on impact to property value:
                        - 1-3: Minor cosmetic issues
                        - 4-6: Moderate damage requiring attention
                        - 7-9: Significant damage affecting value
                        - 10: Critical structural/safety issue"""
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": f"Analyze this property damage image. The file name is: {img_file_name}"
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{base64_image}",
                                    "detail": "high"
                                }
                            }
                        ]
                    }
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": self._damage_schema
                },
                max_tokens=self.max_tokens
            )
            
            # Parse JSON response
            damage_data = json.loads(response.choices[0].message.content)
            output = PropertyDamageOutput(**damage_data)
            
            logger.info(f"Successfully analyzed new image: {img_file_name}")
            return output
            
        except Exception as e:
            logger.error(f"Error analyzing image: {e}")
            raise
    
    def _search_similar_damage(self, query: str, top_k: int = 3) -> VisionSearchResult:
        """
        Search for similar damage cases in Pinecone.
        
        Args:
            query: Text description of damage to search for
            top_k: Number of results to return
            
        Returns:
            VisionSearchResult with multiple matching cases
        """
        try:
            # Create query embedding
            query_embedding = self._create_embedding(query)
            
            # Search Pinecone
            results = self.index.query(
                vector=query_embedding,
                top_k=top_k,
                namespace=self.namespace,
                include_metadata=True
            )
            
            # Parse results into PropertyDamageOutput objects
            damage_cases = []
            for match in results.matches:
                metadata = match.metadata
                
                # Add similarity score to metadata
                damage_output = PropertyDamageOutput(
                    **metadata,
                    similarity_score=round(match.score, 3)
                )
                damage_cases.append(damage_output)
            
            logger.info(f"Found {len(damage_cases)} similar damage cases for query: '{query}'")
            
            return VisionSearchResult(
                query=query,
                results=damage_cases,
                results_count=len(damage_cases)
            )
            
        except Exception as e:
            logger.error(f"Error searching damage cases: {e}")
            raise
    
    def __call__(
        self,
        image_path: Optional[str] = None,
        search_query: Optional[str] = None,
        mode: Literal["analyze", "search"] = "analyze",
        top_k: int = 3
    ):
        """
        Main entry point for the tool.
        
        Args:
            image_path: Path to image (analyze mode)
            search_query: Text query (search mode)
            mode: "analyze" or "search"
            top_k: Number of results (search mode)
            
        Returns:
            PropertyDamageOutput (analyze mode) or VisionSearchResult (search mode)
        """
        logger.info(f"Executing VisionTool in '{mode}' mode")
        
        try:
            if mode == "analyze":
                # Analyze new image
                if not image_path:
                    raise ValueError("image_path is required for analyze mode")
                
                img_path = Path(image_path)
                if not img_path.exists():
                    raise FileNotFoundError(f"Image file not found: {image_path}")
                
                return self._analyze_new_image(img_path)
            
            elif mode == "search":
                # Search existing damage cases
                if not search_query:
                    raise ValueError("search_query is required for search mode")
                
                return self._search_similar_damage(search_query, top_k)
            
            else:
                raise ValueError(f"Invalid mode: {mode}. Must be 'analyze' or 'search'")
                
        except Exception as e:
            logger.error(f"Error executing VisionTool: {e}")
            raise
    
    def get_tool_schema(self) -> dict:
        """Return JSON schema for MCP protocol compatibility."""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": PropertyDamageInput.model_json_schema(),
            "output_schema": {
                "oneOf": [
                    PropertyDamageOutput.model_json_schema(),
                    VisionSearchResult.model_json_schema()
                ]
            }
        }

In [10]:
# Initialize the vision tool
vision_tool = VisionTool()

# Simple damage assessment
result = vision_tool(image_path="../data/prop-damage-images-resized-512/property_images_r3_c5_processed_by_imagy.png")

print(f"Damage Type: {result.damage_type}")
print(f"Affected System: {result.affected_system}")
print(f"Severity: {result.severity}/10")
print(f"Urgency: {result.urgency}")
print(f"Location: {result.location}")
print(f"Estimated Cost: {result.estimated_repair_cost}")
print(f"\nSummary: {result.maintenance_summary}")
print(f"\nSecondary Risks: {result.secondary_damage_risk}")

INFO:__main__:Initializing OpenAI client for vision analysis...
INFO:__main__:Initializing Pinecone client...
INFO:__main__:VisionTool initialized successfully
INFO:__main__:Executing VisionTool in 'analyze' mode
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Successfully analyzed new image: property_images_r3_c5_processed_by_imagy.png


Damage Type: wall stains and missing flooring
Affected System: cosmetic
Severity: 4/10
Urgency: short-term
Location: interior bedroom
Estimated Cost: $500 - $1000

Summary: Repair wall stains, assess and replace flooring; monitor for moisture issues.

Secondary Risks: Potential for mold if moisture issue exists.


In [11]:
# Example 2: Get full output as dict
output_dict = result.model_dump()
print(json.dumps(output_dict, indent=2))

# Example 3: Access tool schema (for MCP registration)
schema = vision_tool.get_tool_schema()
print(json.dumps(schema, indent=2))

{
  "file_name": "property_images_r3_c5_processed_by_imagy.png",
  "damage_type": "wall stains and missing flooring",
  "affected_system": "cosmetic",
  "location": "interior bedroom",
  "severity": 4,
  "urgency": "short-term",
  "visible_area_affected": "medium",
  "estimated_repair_cost": "$500 - $1000",
  "secondary_damage_risk": "Potential for mold if moisture issue exists.",
  "maintenance_summary": "Repair wall stains, assess and replace flooring; monitor for moisture issues.",
  "similarity_score": null
}
{
  "name": "property_damage_assessment",
  "description": "\n    Analyze property damage images or search existing damage assessments.\n\n    Two modes available:\n\n    ANALYZE MODE (default):\n    - Provide an image_path to analyze a new property damage image\n    - Returns detailed damage assessment including severity, cost, urgency\n    - Identifies affected systems (roofing, plumbing, HVAC, etc.)\n\n    SEARCH MODE:\n    - Provide a search_query to find similar damage ca

In [13]:
# Test VisionTool search mode
print("=== TESTING VISION SEARCH MODE ===")
search_results = vision_tool(
    search_query="water damage on ceiling",
    mode="search",
    top_k=5
)

print(f"Found {search_results.results_count} similar cases:\n")
for i, case in enumerate(search_results.results, 1):
    print(f"[{i}] {case.damage_type} - {case.location}")
    print(f"    File: {case.file_name}")  # ← Added this line
    print(f"    Similarity: {case.similarity_score}")
    print(f"    Severity: {case.severity}/10, Cost: {case.estimated_repair_cost}\n")

INFO:__main__:Executing VisionTool in 'search' mode


=== TESTING VISION SEARCH MODE ===


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Found 5 similar damage cases for query: 'water damage on ceiling'


Found 5 similar cases:

[1] Ceiling Water Stain - Bathroom
    File: property_images_r4_c4_processed_by_imagy.png
    Similarity: 0.691
    Severity: 6/10, Cost: $1,000-$3,000

[2] Water Stain - Interior Room
    File: property_images_r3_c5_processed_by_imagy.png
    Similarity: 0.596
    Severity: 4/10, Cost: $300-$800

[3] Ceiling Damage/Mold - Living Room
    File: property_images_r1_c3_processed_by_imagy.png
    Similarity: 0.586
    Severity: 8/10, Cost: $2000-$5000

[4] Mold Growth - Ceiling
    File: property_images_r2_c4_processed_by_imagy.png
    Similarity: 0.574
    Severity: 8/10, Cost: $1500-$3000

[5] Mold Growth - Ceiling
    File: property_images_r1_c2_processed_by_imagy.png
    Similarity: 0.57
    Severity: 8/10, Cost: $1500-$3000

